#  Lab Results : Bronze to Silver Transformation


**Transformations covered (as per mapping sheet) :**
| # | Source Column | Target Column | Type |
|---|---|---|---|
| 1 | lab_result_id | lab_result_id | Deduplication (PK) |
| 2 | loinc_code | loinc_code_std | UPPER + TRIM |
| 3 | test_name | test_name_std | initcap + TRIM |
| 4 | panel_name | panel_name_std | initcap + TRIM |
| 5 | collection_datetime | collection_ts | Multi-format datetime |
| 6 | result_datetime | result_ts | Multi-format datetime |
| 7 | collection_ts + result_ts | tat_hours | Derived TAT |
| 8 | result_value_numeric | result_num_valid | Cast Double |
| 9 | result_value_text | result_text_std | UPPER + TRIM |
| 10 | result_unit | result_unit_std | UPPER + TRIM |
| 11 | reference_range_low | ref_low_valid | Cast Double |
| 12 | reference_range_high | ref_high_valid | Cast Double |
| 13 | result_num + ref_low + ref_high | result_deviation_pct | Derived % deviation |
| 14 | abnormal_flag | abnormal_flag_std | Boolean normalisation |
| 15 | critical_flag | critical_flag_std | Boolean normalisation |
| 16 | result_status | result_status_std | initcap + TRIM; NULL→Unknown |
| 17 | specimen_type | specimen_type_std | UPPER + TRIM |
| 18 | performing_lab | performing_lab_std | initcap + TRIM |
| 19 | (new) | dq_lab_flag | Row-level DQ flag |

## 0. Configuration

## 1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType

## 2. Read Bronze Data

In [0]:
# ── Read source ───────────────────────────────────────────────────────────────
df_bronze_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.lab_results.csv')
)
display(df_bronze_raw.limit(10))
print(f"[BRONZE] Row count : {df_bronze_raw.count()}")
df_bronze_raw.printSchema()

## 3. Deduplication — `lab_result_id` (Primary Key)

In [0]:
# ── Drop duplicate rows based on Primary Key ─────────────────────────────────
df_deduped = df_bronze_raw.dropDuplicates(["lab_result_id"])

print(f"[DEDUP]  Before : {df_bronze_raw.count()}  |  After : {df_deduped.count()}")
display(df_deduped.limit(10))

## 4. Datetime Standardization
### `collection_datetime` → `collection_ts`  and  `result_datetime` → `result_ts`

**Formats observed in data :**
- `dd/MM/yyyy HH:mm`   → 16/04/2020 12:58
- `d/M/yy H:mm`        → 4/6/21 7:44
- `M/d/yy H:mm`        → 4/17/20 3:58  (US short — common in result_datetime)
- `M/d/yy HH:mm`       → 10/30/22 10:37

`F.coalesce` tries each format in order; first non-null parse wins.

In [0]:
# ── collection_datetime → collection_ts ───────────────────────────────────────
df_with_collection_ts = df_deduped.withColumn(
    "collection_ts",
    F.coalesce(
        F.expr("try_to_timestamp(collection_datetime, 'dd/MM/yyyy HH:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'd/M/yy H:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'M/d/yy H:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'M/d/yy HH:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'dd-MM-yyyy HH:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'MM/dd/yyyy HH:mm')"),
        F.expr("try_to_timestamp(collection_datetime, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(collection_datetime, 'yyyy-MM-dd HH:mm')")
    )
)
#display(df_with_collection_ts)

# ── result_datetime → result_ts ───────────────────────────────────────────────
df_with_timestamps = df_with_collection_ts.withColumn(
    "result_ts",
    F.coalesce(
        F.expr("try_to_timestamp(result_datetime, 'M/d/yy H:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'M/d/yy HH:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'dd/MM/yyyy HH:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'd/M/yy H:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'dd-MM-yyyy HH:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'MM/dd/yyyy HH:mm')"),
        F.expr("try_to_timestamp(result_datetime, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(result_datetime, 'yyyy-MM-dd HH:mm')")
    )
)
#display(df_with_timestamps.limit(10))

## 5. Derived Column — Turnaround Time : `tat_hours`

**Formula :** `ROUND( (unix_timestamp(result_ts) - unix_timestamp(collection_ts)) / 3600, 1 )`

NULL when either timestamp is NULL.

In [0]:
df_with_tat = df_with_timestamps.withColumn(
    "tat_hours",
    F.when(
        F.col("collection_ts").isNotNull() & F.col("result_ts").isNotNull(),
        F.round(
            (F.unix_timestamp(F.col("result_ts")) - F.unix_timestamp(F.col("collection_ts")))
            / F.lit(3600.0),
            1
        )
    ).otherwise(F.lit(None).cast(DoubleType()))
)
#display(df_with_tat.limit(10))

## 6. Type Cast & Validation — `result_value_numeric` → `result_num_valid`
Non-numeric source values become NULL after cast.

In [0]:
df_with_result_num = df_with_tat.withColumn(
    "result_num_valid",
    F.col("result_value_numeric").cast(DoubleType())
)
#display(df_with_result_num.limit(10))

## 7. Text Cleansing — `result_value_text` → `result_text_std`
UPPER + TRIM → e.g. POSITIVE / NEGATIVE / DETECTED

In [0]:
df_with_result_text = df_with_result_num.withColumn(
    "result_text_std",
    F.when(
        F.col("result_value_text").isNotNull(),
        F.upper(F.trim(F.col("result_value_text")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_result_text.limit(10))

## 8. Standardization — `result_unit` → `result_unit_std`
UPPER + TRIM → MEQ/L, MG/DL, %

In [0]:
df_with_unit_std = df_with_result_text.withColumn(
    "result_unit_std",
    F.when(
        F.col("result_unit").isNotNull(),
        F.upper(F.trim(F.col("result_unit")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_unit_std.limit(10))

## 9. Standardization — `loinc_code` → `loinc_code_std`
UPPER + TRIM (LOINC canonical format); NULL rows flagged in DQ step.

In [0]:
df_with_loinc_std = df_with_unit_std.withColumn(
    "loinc_code_std",
    F.when(
        F.col("loinc_code").isNotNull(),
        F.upper(F.trim(F.col("loinc_code")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_loinc_std.limit(10))

## 10. Text Cleansing — `test_name` → `test_name_std`
initcap + TRIM (Title Case)

In [0]:
df_with_test_name_std = df_with_loinc_std.withColumn(
    "test_name_std",
    F.when(
        F.col("test_name").isNotNull(),
        F.initcap(F.trim(F.col("test_name")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_test_name_std.limit(10))

## 11. Standardization — `panel_name` → `panel_name_std`
initcap + TRIM (BMP / CBC / LFT etc.)

In [0]:
df_with_panel_std = df_with_test_name_std.withColumn(
    "panel_name_std",
    F.when(
        F.col("panel_name").isNotNull(),
        F.initcap(F.trim(F.col("panel_name")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_panel_std.limit(10))

## 12. Type Cast — Reference Range Columns
`reference_range_low` → `ref_low_valid`  and  `reference_range_high` → `ref_high_valid`

In [0]:
df_with_ref_ranges = (
    df_with_panel_std
    .withColumn("ref_low_valid",  F.col("reference_range_low").cast(DoubleType()))
    .withColumn("ref_high_valid", F.col("reference_range_high").cast(DoubleType()))
)
#display(df_with_panel_std.limit(10))

## 13. Derived — Result Deviation Percentage : `result_deviation_pct`

**Formula :**
```
mid_ref              = (ref_low_valid + ref_high_valid) / 2
result_deviation_pct = ROUND( (result_num_valid - mid_ref) / mid_ref * 100, 2 )
```
NULL when either reference bound or result_num_valid is NULL.
NULL when mid_ref = 0 (guard against divide-by-zero).

In [0]:
df_with_deviation = df_with_ref_ranges.withColumn(
    "result_deviation_pct",
    F.when(
        F.col("ref_low_valid").isNotNull()
        & F.col("ref_high_valid").isNotNull()
        & F.col("result_num_valid").isNotNull()
        & ((F.col("ref_low_valid") + F.col("ref_high_valid")) != F.lit(0.0)),
        F.round(
            (
                F.col("result_num_valid")
                - ((F.col("ref_low_valid") + F.col("ref_high_valid")) / F.lit(2.0))
            )
            / ((F.col("ref_low_valid") + F.col("ref_high_valid")) / F.lit(2.0))
            * F.lit(100.0),
            2
        )
    ).otherwise(F.lit(None).cast(DoubleType()))
)
#display(df_with_deviation.limit(10))

## 14. Boolean Normalisation — `abnormal_flag` → `abnormal_flag_std`

| Source values (case-insensitive) | Target |
|---|---|
| Yes / Y / H / HH / high / L / LL / low / A / Critical / 1 | 1 |
| No / N / normal / Normal / 0 | 0 |
| NULL | 0 (default per mapping) |
| Anything else | NULL |

In [0]:
df_with_abnormal_std = df_with_deviation.withColumn(
    "abnormal_flag_std",
    F.when(
        F.upper(F.trim(F.col("abnormal_flag").cast(StringType()))).isin(
            "YES", "Y", "H", "HH", "HIGH", "L", "LL", "LOW", "A", "CRITICAL", "1"
        ),
        F.lit(1).cast(IntegerType())
    ).when(
        F.upper(F.trim(F.col("abnormal_flag").cast(StringType()))).isin(
            "NO", "N", "NORMAL", "0"
        ),
        F.lit(0).cast(IntegerType())
    ).when(
        F.col("abnormal_flag").isNull(),
        F.lit(0).cast(IntegerType())
    ).otherwise(
        F.lit(None).cast(IntegerType())
    )
)
#display(df_with_abnormal_std.limit(10))

## 15. Boolean Normalisation — `critical_flag` → `critical_flag_std`

| Source values (case-insensitive) | Target |
|---|---|
| Yes / Y / 1 | 1 |
| No / N / 0 | 0 |
| NULL | 0 (default per mapping) |
| Anything else | NULL |

In [0]:
df_with_critical_std = df_with_abnormal_std.withColumn(
    "critical_flag_std",
    F.when(
        F.upper(F.trim(F.col("critical_flag").cast(StringType()))).isin("YES", "Y", "1"),
        F.lit(1).cast(IntegerType())
    ).when(
        F.upper(F.trim(F.col("critical_flag").cast(StringType()))).isin("NO", "N", "0"),
        F.lit(0).cast(IntegerType())
    ).when(
        F.col("critical_flag").isNull(),
        F.lit(0).cast(IntegerType())
    ).otherwise(
        F.lit(None).cast(IntegerType())
    )
)
#display(df_with_critical_std.limit(10))

## 16. Standardization — `result_status` → `result_status_std`
initcap + TRIM ; NULL → 'Unknown'

Observed values: Preliminary, Corrected, final, FINAL, Final, Cancelled → all become initcap.

In [0]:
df_with_status_std = df_with_critical_std.withColumn(
    "result_status_std",
    F.when(
        F.col("result_status").isNotNull(),
        F.initcap(F.trim(F.col("result_status")))
    ).otherwise(F.lit("Unknown"))
)
#display(df_with_status_std.limit(10))

## 17. Standardization — `specimen_type` → `specimen_type_std`
UPPER + TRIM → BLOOD / URINE / CSF / PLASMA / SPUTUM / SERUM etc.

In [0]:
df_with_specimen_std = df_with_status_std.withColumn(
    "specimen_type_std",
    F.when(
        F.col("specimen_type").isNotNull(),
        F.upper(F.trim(F.col("specimen_type")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_specimen_std.limit(10))

## 18. Casing Standardization — `performing_lab` → `performing_lab_std`
initcap + TRIM (Lab name normalization)

In [0]:
df_with_lab_std = df_with_specimen_std.withColumn(
    "performing_lab_std",
    F.when(
        F.col("performing_lab").isNotNull(),
        F.initcap(F.trim(F.col("performing_lab")))
    ).otherwise(F.lit(None).cast(StringType()))
)
#display(df_with_lab_std.limit(10))

## 19. Data Quality Flag — `dq_lab_flag`

**Flag = 1** when any of the following conditions are true :
- `loinc_code` is NULL
- Both `result_num_valid` AND `result_text_std` are NULL (no usable result at all)
- `collection_ts` is NULL (unparseable or missing datetime)

**Default = 0** (row passes all DQ checks)

In [0]:
df_with_dq_flag = df_with_lab_std.withColumn(
    "dq_lab_flag",
    F.when(
        F.col("loinc_code").isNull()
        | (F.col("result_num_valid").isNull() & F.col("result_text_std").isNull())
        | F.col("collection_ts").isNull(),
        F.lit(1).cast(IntegerType())
    ).otherwise(F.lit(0).cast(IntegerType()))
)
#display(df_with_dq_flag.limit(10))

## 20. Select Final Silver Columns

In [0]:
df_silver_lab_results = df_with_dq_flag.select(

    # ── Identifiers (passthrough) ──────────────────────────────────────────
    F.col("lab_result_id"),
    F.col("encounter_id"),
    F.col("patient_id"),
    F.col("order_id"),
    F.col("ordered_by_provider"),
    F.col("ingestion_timestamp"),

    # ── Standardized code / name columns ───────────────────────────────────
    F.col("loinc_code_std"),
    F.col("test_name_std"),
    F.col("panel_name_std"),
    F.col("specimen_type_std"),
    F.col("performing_lab_std"),
    F.col("result_unit_std"),

    # ── Standardized datetime columns ──────────────────────────────────────
    F.col("collection_ts"),
    F.col("result_ts"),

    # ── Derived numeric columns ─────────────────────────────────────────────
    F.col("tat_hours"),
    F.col("result_num_valid"),
    F.col("result_text_std"),
    F.col("ref_low_valid"),
    F.col("ref_high_valid"),
    F.col("result_deviation_pct"),

    # ── Boolean standardized flags ─────────────────────────────────────────
    F.col("abnormal_flag_std"),
    F.col("critical_flag_std"),

    # ── Standardized status ────────────────────────────────────────────────
    F.col("result_status_std"),

    # ── Data Quality flag ──────────────────────────────────────────────────
    F.col("dq_lab_flag")
)

print(f"[SILVER] Final row count : {df_silver_lab_results.count()}")
df_silver_lab_results.printSchema()
display(df_silver_lab_results.limit(10))

## 22. Preview Silver Data

In [0]:
display(df_silver_lab_results.limit(20))

## 23. Write to Silver Layer (Delta)

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "lab_results"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
df_silver_lab_results.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")

## 24. Verify Silver Table (post-write)

In [0]:
# Uncomment after running the write step above

# df_silver_verify = spark.read.format("delta").load(SILVER_DELTA_PATH)
# print(f"[VERIFY] Silver row count : {df_silver_verify.count()}")
# display(df_silver_verify.limit(10))

---
## DataFrame Lineage (Transformation Chain)

```
df_bronze_raw
  └─► df_deduped
        └─► df_with_collection_ts
              └─► df_with_timestamps
                    └─► df_with_tat
                          └─► df_with_result_num
                                └─► df_with_result_text
                                      └─► df_with_unit_std
                                            └─► df_with_loinc_std
                                                  └─► df_with_test_name_std
                                                        └─► df_with_panel_std
                                                              └─► df_with_ref_ranges
                                                                    └─► df_with_deviation
                                                                          └─► df_with_abnormal_std
                                                                                └─► df_with_critical_std
                                                                                      └─► df_with_status_std
                                                                                            └─► df_with_specimen_std
                                                                                                  └─► df_with_lab_std
                                                                                                        └─► df_with_dq_flag
                                                                                                              └─► df_silver_lab_results ✅
```